# Tool Use

The idea behind tool use is to allow the model to access external data that is not available just from training data, like asking how's the weather in your current location.

The flow that tool use takes is as follows:

<img src="../assets/tool_use_chart.png" width="800px" />

So far I've been trying to come up with my own example, derived from what is on Claude Academy. But for the sake of keep things going smoothly I'll use the same project for tool use as in that course.

## Project Overview

This project is simple. Basically, create a reminder. That's it.

For that we'll need a couple of tools:
- Get current datetime
- Add a duration to datetime
- Set a reminder

## Tool Function

This is literally just a Python function.

In [3]:
from datetime import datetime

def get_current_datetime(format="%Y-%m-%d %H:%M:%S"):
    now = datetime.now()
    return now.strftime(format)

get_current_datetime()

'2026-09-08 22:02:22'


## Tool Schema

Now we need to create the schema that makes the model understand how to call and what each parameter should be.

The schema for Claude SDK is different than LiteLLM. These is what I found for LiteLLM: https://docs.litellm.ai/docs/completion/function_call

In [4]:
get_current_datetime_schema = {
    "type": "function",
    "function": {
        "name": "get_current_datetime",
        "description": "Get the current datetime in a given format",
        "parameters": {
            "type": "object",
            "properties": {
                "format": {
                    "type": "string",
                    "description": "A string with the desired output format, e.g. \"%Y-%m-%d %H:%M:%S\".",
                },
            },
            "required": [],
        },
    },
}

This one I wrote manually, but the course emphasizes that we can ask a model to generate this schema based on the function code.

It's interesting to note that the whole "parameters" could be come from a Pydantic model, using `.model_json_schema()`.

In [5]:
from constants import MODEL
from litellm import Message, completion, supports_function_calling

supports_function_calling(model=MODEL)

True

In [6]:
response = completion(
    model=MODEL,
    messages=[Message(role="user", content="What time it is?")],
    tools=[get_current_datetime_schema],
    tool_choice="auto"
)

response.choices

[Choices(finish_reason='stop', index=0, message=Message(content='{}', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None))]

In [9]:
from constants import CHAT_MODEL


response = completion(
    model=CHAT_MODEL,
    messages=[Message(role="user", content="Hi there! I wanted to know what time it is.")],
    tools=[get_current_datetime_schema],
    tool_choice="auto"
)

response.choices

[Choices(finish_reason='tool_calls', index=0, message=Message(content='', role='assistant', tool_calls=[ChatCompletionMessageToolCall(function=Function(arguments='{"format": "%Y-%m-%d %H:%M:%S"}', name='get_current_datetime'), id='call_ct1g15yb', type='function')], function_call=None, reasoning_content='Okay, the user asked for the current time. I need to use the get_current_datetime function. The function requires a format string. Let me check the parameters. The format example is "%Y-%m-%d %H:%M:%S". I should return the current datetime in that format. Let me make sure to call the function correctly. No other parameters are needed. Just pass the format as specified. Alright, I\'ll generate the tool call with the format string.\n', provider_specific_fields=None))]

Ok so the first example didn't work. Apparently there there are two different prefixes that LiteLLM support:

| Prefix | Routes to  |
| :---   | ---:       |
| ollama/| /api/generate |
| ollama_chat/ | /api/chat

So `/api/chat` is the correct we want to be using, since it supports tool calling and maybe even prefilling (I plan on revisiting the [prefilling section](llm-prompting.ipynb#structure-data))

In [12]:
import json

tool_calls = response.choices[0].message.tool_calls
for tool_call in tool_calls:
    func = globals()[tool_call.function.name]
    params = tool_call.function.arguments
    print(func(**json.loads(params)))

2026-09-08 23:01:52


So now we'll need to loop this function execution back into the model.

To make it easier to control the flow, I'll create a class to handle the tools an another to handle the messages.

In [ ]:
import json
from datetime import datetime
from litellm import completion, Message, ModelResponse
from constants import CHAT_MODEL
from typing import overload, Union

class ToolHandler:
    def __init__(self):
        self.available_tools = [
            {
                "type": "function",
                "function": {
                    "name": "get_current_datetime",
                    "description": "Get the current datetime in a given format",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "format": {
                                "type": "string",
                                "description": "A string with the desired output format, e.g. \"%Y-%m-%d %H:%M:%S\".",
                            },
                        },
                        "required": [],
                    },
                },
            }
        ]

    def get_current_datetime(self, format: str = "%Y-%m-%d %H:%M:%S"):
        now = datetime.now()
        return now.strftime(format)

class LLM:
    def __init__(self, model = CHAT_MODEL):
        self.model = model
        self.tool_handler = ToolHandler()
        self.messages: list[Message] = []

    @overload
    def add_message(self, message: Message) -> None: ...

    @overload
    def add_message(self, content: str, role: str = "user") -> None: ...

    def add_message(self, message_or_content: Union[Message, str], role: str = "user") -> None:
        if isinstance(message_or_content, Message):
            self.messages.append(message_or_content)

        elif isinstance(message_or_content, str):
            self.messages.append(Message(role=role, content=message_or_content))

    def send(self, prompt: str = None):
        if prompt:
            self.add_message(prompt)

        response = completion(
            model=self.model,
            messages=self.messages,
            tools=self.tool_handler.available_tools
        )
        self.__handle_response(response)

    def __handle_response(self, response: ModelResponse):
        message = response.choices[0].message
        self.add_message(message)

        if message.content:
            print(message.content)

        if message.tool_calls:
            for call in message.tool_calls:
                func = getattr(self.tool_handler, call.function.name)
                ret = func(**json.loads(call.function.arguments))
                self.add_message(Message(role="tool", name=call.function.name, tool_call_id=call.id, content=ret))
            self.send()

In [22]:
llm = LLM()

llm.send("What time is it?")

The current time is 11:38 PM on September 8, 2026.


I wonder if I can get JSON schema based on the function alone. Get the types from python type hinting and field description from docstrings. That'd be nice.

Apparently there's a lib that does that, but there's also the option to use Pydantic. It's absolutely optional, but I want to try it out.

In [38]:
import json
import inspect
from datetime import datetime, timedelta
from litellm import completion, Message, ModelResponse
from constants import CHAT_MODEL
from typing import overload, Union, Annotated
from pydantic import TypeAdapter, Field

class ToolHandler:
    def __init__(self):
        self.available_tools = []

        for name, method in inspect.getmembers(self, predicate=inspect.ismethod):
            if name.startswith("__"):
                continue
            param_schema = TypeAdapter(method).json_schema()
            method_description = method.__doc__.strip() if method.__doc__ else "No description."
            self.available_tools.append(
                {
                    "type": "function",
                    "function": {
                        "name": name,
                        "description": method_description,
                        "parameters": param_schema,
                    },
                },
            )

    def get_current_datetime(
            self,
            format: Annotated[str, Field(description="A string with the desired output format, e.g. \"%Y-%m-%d %H:%M:%S\".")] = "%Y-%m-%d %H:%M:%S"
        ):
        """Get the current datetime in a given format"""
        now = datetime.now()
        return now.strftime(format)

class LLM:
    def __init__(self, model = CHAT_MODEL, tool_handler: ToolHandler = ToolHandler(), debug: bool = False):
        self.model = model
        self.debug = debug
        self.tool_handler = tool_handler
        self.messages: list[Message] = []

    @overload
    def add_message(self, message: Message) -> None: ...

    @overload
    def add_message(self, content: str, role: str = "user") -> None: ...

    def add_message(self, message_or_content: Union[Message, str], role: str = "user") -> None:
        if isinstance(message_or_content, Message):
            self.messages.append(message_or_content)

        elif isinstance(message_or_content, str):
            self.messages.append(Message(role=role, content=message_or_content))

    def send(self, prompt: str = None):
        if prompt:
            self.add_message(prompt)

        response = completion(
            model=self.model,
            messages=self.messages,
            tools=self.tool_handler.available_tools
        )
        self.__handle_response(response)

    def __handle_response(self, response: ModelResponse):
        message = response.choices[0].message
        self.add_message(message)

        if self.debug:
            print(self.messages)

        if message.content:
            print(message.content)

        if message.tool_calls:
            for call in message.tool_calls:
                func = getattr(self.tool_handler, call.function.name)
                ret = func(**json.loads(call.function.arguments))
                self.add_message(Message(role="tool", name=call.function.name, tool_call_id=call.id, content=ret))
            self.send()

In [32]:
llm = LLM()

llm.send("What time is it?")

The current date and time is **2026-09-09 00:12:27**. Let me know if you need further assistance!


Works like a charm.

The next chapter cover basically what I've already done:

- Sending the return of the tools
- Handling multiple sequential request for tools usage

The key difference is that the way I did it's recursive:
1. Send the message
2. Get response and send it to handler
3. If the response has a tool call, call the tool and send message back (back to step 1)

I also already handle multiple tool calls from a single message.

In [39]:

class ToolHandler:
    def __init__(self):
        self.available_tools = []

        for name, method in inspect.getmembers(self, predicate=inspect.ismethod):
            if name.startswith("__"):
                continue
            param_schema = TypeAdapter(method).json_schema()
            method_description = method.__doc__.strip() if method.__doc__ else "No description."
            self.available_tools.append(
                {
                    "type": "function",
                    "function": {
                        "name": name,
                        "description": method_description,
                        "parameters": param_schema,
                    },
                },
            )


class MyCustomHandler(ToolHandler):
    def get_current_datetime(
        self,
        format: Annotated[str, Field(description="A string with the desired output format, e.g. \"%Y-%m-%d %H:%M:%S\".")] = "%Y-%m-%d %H:%M:%S"
    ):
        """Get the current datetime in a given format"""
        now = datetime.now()
        return now.strftime(format)

    def add_delta_to_datetime(
        self,
        base_datetime: Annotated[str, Field(description="A string with the base datetime, in \"%Y-%m-%d %H:%M:%S\" format, to add the delta to.")],
        days: Annotated[int, Field(description="Days to add to the datetime.")] = 0,
    ):
        """Adds a timedelta to the base datetime provided"""
        _datetime = datetime.strptime(base_datetime, "%Y-%m-%d %H:%M:%S")
        new_datetime = _datetime + timedelta(days=days)
        return new_datetime.strftime("%Y-%m-%d %H:%M:%S")

In [41]:
llm = LLM(tool_handler=MyCustomHandler(), debug=True)

llm.send("Whats the date 10 days from now?")

[Message(content='Whats the date 10 days from now?', role='user', tool_calls=None, function_call=None, provider_specific_fields=None), Message(content='', role='assistant', tool_calls=[ChatCompletionMessageToolCall(function=Function(arguments='{"format": "%Y-%m-%d %H:%M:%S"}', name='get_current_datetime'), id='call_x3x4x0f8', type='function')], function_call=None, reasoning_content='Okay, the user is asking for the date 10 days from now. Let me figure out how to handle this.\n\nFirst, I need to get the current datetime. The available functions include get_current_datetime, which can retrieve the current date and time in a specified format. Since the user didn\'t mention a specific format, but the add_delta_to_datetime function expects a base_datetime in "%Y-%m-%d %H:%M:%S" format, I should use that format when getting the current datetime.\n\nSo, I\'ll call get_current_datetime with the format "%Y-%m-%d %H:%M:%S". That will give me the current date and time as a string in the required 

Behaviour is kinda erratic on this one.
Sometimes it calls one tool at a time, sometimes it call both at once with a hallucinated datetime as base to offset...
Sometimes the final response is in `content` the answer is on `reasoning_content`.

In [6]:
import json
import inspect
from litellm import completion, Message, ModelResponse
from constants import CHAT_MODEL
from typing import overload, Union
from pydantic import TypeAdapter


class ToolHandler:
    def __init__(self):
        self.available_tools = []

        for name, method in inspect.getmembers(self, predicate=inspect.ismethod):
            if name.startswith("__"):
                continue
            param_schema = TypeAdapter(method).json_schema()
            method_description = method.__doc__.strip() if method.__doc__ else "No description."
            self.available_tools.append(
                {
                    "type": "function",
                    "function": {
                        "name": name,
                        "description": method_description,
                        "parameters": param_schema,
                    },
                },
            )


class LLM:
    def __init__(self, model = CHAT_MODEL, tool_handler: ToolHandler = ToolHandler(), debug: bool = False):
        self.model = model
        self.debug = debug
        self.tool_handler = tool_handler
        self.messages: list[Message] = []

    @overload
    def add_message(self, message: Message) -> None: ...

    @overload
    def add_message(self, content: str, role: str = "user") -> None: ...

    def add_message(self, message_or_content: Union[Message, str], role: str = "user") -> None:
        if isinstance(message_or_content, Message):
            self.messages.append(message_or_content)

        elif isinstance(message_or_content, str):
            self.messages.append(Message(role=role, content=message_or_content))

    def send(self, prompt: str = None):
        if prompt:
            self.add_message(prompt)

        response = completion(
            model=self.model,
            messages=self.messages,
            tools=self.tool_handler.available_tools
        )
        self.__handle_response(response)

    def __handle_response(self, response: ModelResponse):
        message = response.choices[0].message
        self.add_message(message)

        if self.debug:
            print(self.messages)

        

        if not message.tool_calls:
            print(message.content or message.reasoning_content)

        if message.tool_calls:
            if message.content:
                print(message.content)
            for call in message.tool_calls:
                func = getattr(self.tool_handler, call.function.name)
                ret = func(**json.loads(call.function.arguments))
                self.add_message(Message(role="tool", name=call.function.name, tool_call_id=call.id, content=ret))
            self.send()

In [7]:
from datetime import datetime, timedelta
from typing import Annotated
from pydantic import Field

class MyCustomHandler(ToolHandler):
    def get_current_datetime(
        self,
        format: Annotated[str, Field(description="A string with the desired output format, e.g. \"%Y-%m-%d %H:%M:%S\".")] = "%Y-%m-%d %H:%M:%S"
    ):
        """Get the current datetime in a given format"""
        now = datetime.now()
        return now.strftime(format)

    def add_delta_to_datetime(
        self,
        base_datetime: Annotated[str, Field(description="A string with the base datetime, in \"%Y-%m-%d %H:%M:%S\" format, to add the delta to.")],
        days: Annotated[int, Field(description="Days to add to the datetime.")] = 0,
    ):
        """Adds a timedelta to the base datetime provided"""
        _datetime = datetime.strptime(base_datetime, "%Y-%m-%d %H:%M:%S")
        new_datetime = _datetime + timedelta(days=days)
        return new_datetime.strftime("%Y-%m-%d %H:%M:%S")

In [8]:
llm = LLM(tool_handler=MyCustomHandler())

llm.send("Whats the date 10 days from now?")

The date 10 days from now, September 9, 2026, is **September 19, 2026**.
